# MISSION: STUDIO 555 — JAZZ'S PERSONAL AI VOICE (FREE Colab)

This Google Colab notebook shows step-by-step, beginner-friendly instructions to clone *your own voice only* using the open-source Chatterbox code and the lightweight Chatterbox-Nano model. It tries to run on free Colab resources when possible and keeps you in control of uploads and downloads.

IMPORTANT NOTES:
- This uses only open-source code (no paid APIs).
- You must upload a *clean 10–20 second* recording of *your own* voice. Do not upload other people’s or copyrighted/celebrity voices.
- If you do not upload a reference clip, the notebook will stop and show an error (it will NOT silently fall back to a demo voice).
- Your uploaded audio and generated files are processed on Google Colab (Google Cloud). They are NOT added to this GitHub repository. Do NOT paste secrets or API keys into the notebook.

If the notebook cannot be saved to your GitHub (permission error), the final cell below explains how to save and upload this notebook from your phone using GitHub mobile or directly open it in Colab via the provided link.

---
## 1) Install runtime dependencies (runs in Colab)
Tap ▶ to run this cell. It installs ffmpeg (for audio conversion), the open-source Chatterbox python package from this repository, and common audio libraries. This may take a few minutes the first time.


In [ ]:
# Install system and python deps. This is safe and free.
import sys
print('Python', sys.version)
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install --upgrade pip --quiet
# Install the open-source chatterbox from the repository (free).
!pip install --quiet git+https://github.com/resemble-ai/chatterbox.git@master
!pip install --quiet soundfile librosa==0.11.0
print('Installed packages.')


---
## 2) Upload your clean 10–20s reference clip (WAV, MP3, M4A ok)
Tap ▶ to run the cell, then use the file chooser to pick a file from your Android phone.
*Guidance:* Record in a quiet room, hold the phone close, use the phone's voice memo app. Save as WAV if possible; MP3/M4A also accepted and will be converted.


In [ ]:
from google.colab import files
print('Tap the file chooser that appears to upload your 10-20s reference voice file (WAV/MP3/M4A).')
uploaded = files.upload()
if len(uploaded) == 0:
    raise FileNotFoundError('No file uploaded. Please run the cell again and choose your reference audio file.')
ref_filename = list(uploaded.keys())[0]
print('Uploaded:', ref_filename)


---
## 3) Load the lightweight Chatterbox-Nano model (CPU-friendly)
This loads Nano (110M). It still downloads model weights (~tens to hundreds of MB). Be patient — on free Colab this may take several minutes.


In [ ]:
import torch
print('Torch version:', torch.__version__)
from chatterbox.tts_turbo import ChatterboxTurboTTS
device = 'cpu'  # Free Colab: use CPU to avoid CUDA requirements. If you have a paid GPU runtime you may change this.
print('Loading Chatterbox (nano=True) on', device, ' — this may take a few minutes while weights download.')
model = ChatterboxTurboTTS.from_pretrained(device=device, nano=True)
print('Model loaded. Sample rate (sr) =', model.sr)


---
## 4) Convert your uploaded reference to the model's required WAV format
This will resample and save a clean 16-bit PCM WAV at the model's sample rate. If no reference file was uploaded, the cell will error and stop — this prevents silent fallbacks to demo voices.


In [ ]:
import os
import librosa
import soundfile as sf
# Ensure uploaded ref exists
if 'ref_filename' not in globals():
    raise FileNotFoundError('Reference audio not found. Run the upload cell and upload a 10-20s clip of your own voice.')
infile = ref_filename
print('Reading uploaded file:', infile)
# Load and resample to model.sr (mono)
audio, orig_sr = librosa.load(infile, sr=model.sr, mono=True)
out_ref = 'ref_converted.wav'
# Save as 16-bit PCM WAV
sf.write(out_ref, audio, model.sr, subtype='PCM_16')
print('Saved converted reference as', out_ref, ' (sr=', model.sr, ')')
print('Reference duration (s):', len(audio)/model.sr)
if len(audio)/model.sr < 6 or len(audio)/model.sr > 30:
    print('Warning: recommended reference length is ~10-20 seconds for best cloning quality.')


---
## 5) Type the exact words you want the model to speak
Enter the exact phrase below and run the cell. Keep it short to start (a few seconds).


In [ ]:
from IPython.display import Audio, display
from google.colab import files as gcfiles
import torch
# Double-check ref exists
if not os.path.exists('ref_converted.wav'):
    raise FileNotFoundError('Converted reference ref_converted.wav not found. Run the conversion cell first.')
prompt_text = input("Type the exact words you want spoken: ")
if len(prompt_text.strip()) == 0:
    raise ValueError('No text provided — please type the exact words you want spoken.')
print('Generating audio for:', prompt_text)
try:
    out = model.generate(prompt_text, audio_prompt_path='ref_converted.wav')
    # model.generate may return a torch tensor or numpy array
    if isinstance(out, torch.Tensor):
        out = out.cpu().numpy()
    # If multi-channel, take first channel or flatten
    import numpy as np
    if hasattr(out, 'ndim') and out.ndim > 1:
        out = np.mean(out, axis=0)
    sf.write('generated.wav', out, model.sr, subtype='PCM_16')
    print('Saved generated.wav')
    display(Audio('generated.wav', rate=model.sr))
    print('If you want to download the file to your phone, the next command will start the download.')
    gcfiles.download('generated.wav')
except Exception as e:
    print('Generation failed with an error:')
    print(str(e))
    print('\nCommon causes: not enough free RAM, network error while downloading model weights, or incompatible runtime. Try re-running the model-load cell, switch to a GPU runtime in Colab (Runtime > Change runtime type > GPU).')


---
## 6) Privacy & storage — IMPORTANT
- Files you upload here are processed on Google Colab (Google Cloud). They are NOT automatically pushed to GitHub.
- Do NOT paste API keys, secrets, or personal data into the notebook cells.
- If you want to keep generated audio private, download it from this Colab session to your phone and then delete the Colab runtime (Runtime > Manage sessions > terminate).


---
## 7) What I (the author) did and what I couldn't test here
- This notebook was added to your repository as a Colab-ready example that installs Chatterbox from the public repo and uses the Nano variant.
- I validated the notebook structure and used Colab-friendly methods (google.colab.files, IPython.display.Audio).
- I could NOT run the notebook in your Colab session from here — model weight downloads and runtime-specific behaviour (free Colab vs paid GPU) depend on your Google account and runtime selection.
- If anything fails during model.load or generate, the printed error message will help. Common fixes: re-run the model-load cell, switch to a GPU runtime, or increase session memory.


---
## 8) If I can't write to your GitHub (permissions) — how to save this notebook from your phone
If the notebook file does not appear in your repo (permission error or 403), do this: 
1) Open this repository in your phone browser: https://github.com/jazmindragon95-hub/chatterbox
2) Tap the green `Add file` button (or `Create new file`) under the repository file list.
3) For the filename enter: `notebooks/STUDIO555_Jazz_Own_Voice_FREE_Colab.ipynb`
4) Switch to the Colab notebook view by opening Google Colab on your phone and creating a new notebook. Copy the raw JSON content of this notebook (you can view it by opening the notebook file in GitHub on a desktop or by copying from this page) and paste it into the GitHub file editor. Commit the file.
OR simpler: open the direct Colab link provided below, then in Colab choose File > Save a copy in GitHub and follow prompts to save to your repo/branch/path.


---
## Android step-by-step (exact taps) — how to use this from your phone
Follow these steps on your Android phone (I'll assume Chrome):
1) Open Chrome and go to: https://colab.research.google.com/github/jazmindragon95-hub/chatterbox/blob/master/notebooks/STUDIO555_Jazz_Own_Voice_FREE_Colab.ipynb
2) If asked, sign into your Google account.
3) In Colab, tap the Play ▶ icons on each cell in order (Install → Upload → Load model → Convert → Generate).
4) When uploading: Colab will open a file chooser — select your recorded 10–20s clip from your phone (e.g., in Files or in the Voice Recorder app export).
5) Wait for the model to load (this may take several minutes).
6) Type the exact text you want to synthesize when prompted, and allow the download to complete.
7) The generated.wav will be offered as a download — accept it and save to your phone.
8) When finished, in Colab go to Runtime > Manage sessions and terminate the session to remove files from the cloud.
